# 04_analisis_exploratorio_formal

## Objetivo
Evaluar el volumen, composición, temporalidad y calidad del corpus formal `media_anchored` de HateCR después de la limpieza del notebook 03.

## Entradas
- `data/processed/x_media_anchored_interactions_corpus_formal_analysis.csv`
- `data/processed/x_media_anchored_interactions_corpus_formal_clean.csv`
- `data/processed/x_media_anchored_interactions_corpus_formal_training_dedup.csv`
- `data/interim/source_posts_formal_unique.csv`
- `data/interim/reply_collection_manifest_formal.csv`
- `data/interim/formal_collection/batch_*/replies_stats.csv`
- `lexicons/processed/hatecr_lexicon.csv`, si está disponible

## Salidas
- tablas en `reports/formal_eda/`
- figuras en `reports/figures/formal_eda/`
- `data/processed/x_media_anchored_interactions_corpus_formal_eda.csv`

El notebook no llama a la API, no modifica datos de recolección y no sobrescribe los reportes ni etiquetas del piloto anterior.

## 1. Parámetros y setup

In [ ]:
import os
import sys
import importlib
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display


def find_project_root(start):
    for candidate in [start] + list(start.parents):
        if (candidate / "config").exists() and (candidate / "src").exists():
            return candidate
        child = candidate / "HateCR"
        if (child / "config").exists() and (child / "src").exists():
            return child
    raise FileNotFoundError("No se encontró la raíz del proyecto HateCR")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.eda as eda
importlib.reload(eda)

DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FORMAL_COLLECTION_DIR = DATA_INTERIM / "formal_collection"
REPORTS_DIR = PROJECT_ROOT / "reports" / "formal_eda"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures" / "formal_eda"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

CORPUS_PATH = DATA_PROCESSED / "x_media_anchored_interactions_corpus_formal_analysis.csv"
MASTER_PATH = DATA_PROCESSED / "x_media_anchored_interactions_corpus_formal_clean.csv"
TRAINING_PATH = DATA_PROCESSED / "x_media_anchored_interactions_corpus_formal_training_dedup.csv"
SOURCE_POSTS_PATH = DATA_INTERIM / "source_posts_formal_unique.csv"
REPLY_MANIFEST_PATH = DATA_INTERIM / "reply_collection_manifest_formal.csv"
LEXICON_PATH = PROJECT_ROOT / "lexicons" / "processed" / "hatecr_lexicon.csv"
EDA_OUTPUT_PATH = DATA_PROCESSED / "x_media_anchored_interactions_corpus_formal_eda.csv"

MANUAL_SAMPLE_PER_EVENT = int(os.getenv("MANUAL_SAMPLE_PER_EVENT", "30"))
RANDOM_STATE = int(os.getenv("RANDOM_STATE", "42"))
TOP_N = int(os.getenv("EDA_TOP_N", "100"))

COLORS = {
    "navy": "#12355B",
    "teal": "#087E8B",
    "red": "#D1495B",
    "gold": "#E9B949",
    "gray": "#667085",
}
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.grid": True,
    "grid.alpha": 0.2,
    "font.family": "DejaVu Serif",
    "axes.titleweight": "bold",
})

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CORPUS_PATH:", CORPUS_PATH)
print("API de X: desactivada")

## 2. Carga segura

In [ ]:
id_dtypes = {
    "tweet_id": "string",
    "reply_id": "string",
    "source_post_id": "string",
    "anchor_post_id": "string",
    "conversation_id": "string",
    "reply_author_id_hash": "string",
}
corpus_df = eda.safe_read_csv(CORPUS_PATH, "corpus formal analítico", dtype=id_dtypes)
master_df = eda.safe_read_csv(MASTER_PATH, "corpus formal maestro", dtype=id_dtypes)
training_df = eda.safe_read_csv(TRAINING_PATH, "corpus formal entrenamiento", dtype=id_dtypes)
source_posts_df = eda.safe_read_csv(
    SOURCE_POSTS_PATH,
    "posts madre formales",
    dtype={"source_post_id": "string", "tweet_id": "string"},
)
reply_manifest_df = eda.safe_read_csv(
    REPLY_MANIFEST_PATH,
    "manifiesto formal de replies",
    dtype={"source_post_id": "string"},
)
reply_stats_df = eda.load_formal_reply_stats(FORMAL_COLLECTION_DIR)
lexicon_df = eda.safe_read_csv(LEXICON_PATH, "lexicón HateCR", dtype=str)

availability_df = pd.DataFrame([
    {"dataset": "corpus_analysis", "rows": len(corpus_df)},
    {"dataset": "corpus_master", "rows": len(master_df)},
    {"dataset": "corpus_training_dedup", "rows": len(training_df)},
    {"dataset": "source_posts", "rows": len(source_posts_df)},
    {"dataset": "reply_manifest", "rows": len(reply_manifest_df)},
    {"dataset": "reply_stats", "rows": len(reply_stats_df)},
    {"dataset": "lexicon", "rows": len(lexicon_df)},
])
display(availability_df)

if corpus_df.empty:
    raise ValueError("No existe corpus formal analítico. Ejecuta primero el notebook 03.")

## 3. Validaciones metodológicas

In [ ]:
def as_bool(series):
    return series.fillna("").astype(str).str.strip().str.lower().isin(
        {"1", "true", "yes", "si", "s"}
    )


required = [
    "tweet_id", "text", "text_norm", "text_norm_no_accents",
    "reply_author_id_hash", "source_type", "source_universe",
    "anchor_post_id", "anchor_media_handle", "event_id",
    "created_at", "lang", "eligible_for_text_analysis",
]
missing = [column for column in required if column not in corpus_df.columns]
assert not missing, f"Faltan columnas: {missing}"
assert corpus_df["tweet_id"].notna().all()
assert corpus_df["tweet_id"].is_unique
assert corpus_df["source_type"].eq("reply_to_media_post").all()
assert corpus_df["source_universe"].eq("media_anchored").all()
assert corpus_df["anchor_media_handle"].notna().all()
assert corpus_df["reply_author_id_hash"].str.fullmatch(r"[0-9a-f]{64}").all()
assert as_bool(corpus_df["eligible_for_text_analysis"]).all()
assert corpus_df["event_id"].nunique() == 6
assert not {"author_id", "username", "screen_name"}.intersection(corpus_df.columns)

with open(PROJECT_ROOT / "config" / "events.yaml", encoding="utf-8") as handle:
    events_cfg = yaml.safe_load(handle) or {}
formal_events = sorted(
    [event for event in events_cfg.get("events", []) if event.get("formal", False)],
    key=lambda event: int(event.get("formal_order", 999)),
)
EVENT_ORDER = [str(event["event_id"]) for event in formal_events]
EVENT_NAMES = {str(event["event_id"]): str(event["event_name"]) for event in formal_events}

for metric in ["reply_count", "quote_count", "retweet_count", "like_count", "impression_count"]:
    if metric not in corpus_df.columns:
        corpus_df[metric] = 0
    corpus_df[metric] = pd.to_numeric(corpus_df[metric], errors="coerce").fillna(0)

corpus_df["created_at_dt"] = pd.to_datetime(corpus_df["created_at"], errors="coerce", utc=True)
corpus_df["created_at_cr"] = corpus_df["created_at_dt"].dt.tz_convert("America/Costa_Rica")
corpus_df["date_cr"] = corpus_df["created_at_cr"].dt.date
corpus_df["hour_cr"] = corpus_df["created_at_cr"].dt.hour

print("[OK] Corpus formal media-anchored validado")
print("Replies:", len(corpus_df))
print("Autores anonimizados:", corpus_df["reply_author_id_hash"].nunique())
print("Eventos:", corpus_df["event_id"].nunique())
print("Medios ancla:", corpus_df["anchor_media_handle"].nunique())

## 4. Resumen general y composición

In [ ]:
by_event_df = eda.summarize_by_group(corpus_df, ["event_id", "event_name"])
by_event_df["event_order"] = by_event_df["event_id"].map(
    {event_id: index for index, event_id in enumerate(EVENT_ORDER)}
)
by_event_df = by_event_df.sort_values("event_order").reset_index(drop=True)

by_media_df = eda.summarize_by_group(
    corpus_df, ["anchor_media_id", "anchor_media_handle"]
)
by_batch_df = eda.summarize_by_group(
    corpus_df, ["collection_batch_id", "collection_batch_status"]
)
by_lang_df = eda.summarize_by_group(corpus_df, ["lang", "lang_group"])
by_event_media_df = eda.summarize_by_group(
    corpus_df, ["event_id", "anchor_media_id", "anchor_media_handle"]
)
by_source_type_df = eda.summarize_by_group(corpus_df, "source_type")

summary_general_df = pd.DataFrame([
    {"metric": "master_rows", "value": len(master_df)},
    {"metric": "analysis_rows", "value": len(corpus_df)},
    {"metric": "training_dedup_rows", "value": len(training_df)},
    {"metric": "unique_tweets", "value": corpus_df["tweet_id"].nunique()},
    {"metric": "unique_author_hashes", "value": corpus_df["reply_author_id_hash"].nunique()},
    {"metric": "formal_events", "value": corpus_df["event_id"].nunique()},
    {"metric": "anchor_media", "value": corpus_df["anchor_media_handle"].nunique()},
    {"metric": "source_types", "value": "|".join(sorted(corpus_df["source_type"].unique()))},
    {"metric": "min_created_at_cr", "value": str(corpus_df["created_at_cr"].min())},
    {"metric": "max_created_at_cr", "value": str(corpus_df["created_at_cr"].max())},
    {"metric": "duplicate_tweet_ids", "value": int(corpus_df["tweet_id"].duplicated().sum())},
])

tables = {
    "eda_summary_general.csv": summary_general_df,
    "eda_by_event.csv": by_event_df,
    "eda_by_anchor_media.csv": by_media_df,
    "eda_by_batch.csv": by_batch_df,
    "eda_by_lang.csv": by_lang_df,
    "eda_by_event_media.csv": by_event_media_df,
    "eda_by_source_type.csv": by_source_type_df,
}
for filename, table in tables.items():
    eda.atomic_to_csv(table, REPORTS_DIR / filename)

display(summary_general_df)
display(by_event_df)
display(by_media_df.head(30))

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(by_event_df["event_name"], by_event_df["n_rows"], color=COLORS["navy"])
ax.set_title("Replies analizables por momento formal")
ax.set_ylabel("Replies")
ax.tick_params(axis="x", rotation=35)
for label in ax.get_xticklabels():
    label.set_ha("right")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "volume_by_event.png", dpi=170)
plt.close(fig)

media_plot = by_media_df.sort_values("n_rows").tail(20)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(media_plot["anchor_media_handle"], media_plot["n_rows"], color=COLORS["teal"])
ax.set_title("Interacciones por medio ancla")
ax.set_xlabel("Replies analizables")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "top_anchor_media.png", dpi=170)
plt.close(fig)

## 5. Distribución temporal en hora de Costa Rica

In [ ]:
tweets_by_event_date_df = (
    corpus_df.groupby(["event_id", "event_name", "date_cr"], dropna=False)
    .size().reset_index(name="n_replies")
)
tweets_by_hour_cr_df = (
    corpus_df.groupby("hour_cr", dropna=False)
    .size().reset_index(name="n_replies")
    .sort_values("hour_cr")
)
eda.atomic_to_csv(tweets_by_event_date_df, REPORTS_DIR / "eda_by_event_date_cr.csv")
eda.atomic_to_csv(tweets_by_hour_cr_df, REPORTS_DIR / "eda_by_hour_cr.csv")

fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
for axis, event_id in zip(axes.flat, EVENT_ORDER):
    part = tweets_by_event_date_df[tweets_by_event_date_df["event_id"].eq(event_id)].sort_values("date_cr")
    axis.plot(part["date_cr"].astype(str), part["n_replies"], marker="o", color=COLORS["red"], linewidth=2)
    axis.set_title(EVENT_NAMES.get(event_id, event_id), fontsize=10)
    axis.tick_params(axis="x", rotation=35, labelsize=8)
    axis.set_ylabel("Replies")
fig.suptitle("Distribución diaria dentro de cada momento formal", fontsize=15, fontweight="bold")
fig.savefig(FIGURES_DIR / "event_daily_timeline_cr.png", dpi=170)
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(tweets_by_hour_cr_df["hour_cr"].astype(int), tweets_by_hour_cr_df["n_replies"], color=COLORS["gold"])
ax.set_title("Replies por hora del día, hora de Costa Rica")
ax.set_xlabel("Hora")
ax.set_ylabel("Replies")
ax.set_xticks(range(0, 24))
fig.tight_layout()
fig.savefig(FIGURES_DIR / "replies_by_hour_cr.png", dpi=170)
plt.close(fig)

display(tweets_by_event_date_df)

## 6. Posts madre y cobertura de recolección

In [ ]:
if not reply_manifest_df.empty:
    selected_mask = as_bool(reply_manifest_df["selected_for_collection"])
    selected_manifest_df = reply_manifest_df[selected_mask].copy()
else:
    selected_manifest_df = pd.DataFrame()

if not source_posts_df.empty:
    for metric in ["reply_count", "quote_count", "retweet_count", "like_count", "engagement_score"]:
        if metric not in source_posts_df.columns:
            source_posts_df[metric] = 0
        source_posts_df[metric] = pd.to_numeric(source_posts_df[metric], errors="coerce").fillna(0)

actual_replies_df = (
    master_df.groupby("source_post_id", dropna=False)
    .agg(
        n_replies_collected=("tweet_id", "nunique"),
        n_reply_authors=("reply_author_id_hash", "nunique"),
    )
    .reset_index()
)

if not selected_manifest_df.empty:
    coverage_cols = [
        "collection_batch_id", "source_post_id", "source_post_url",
        "event_id", "event_name", "formal_event_memberships",
        "media_id", "media_name", "media_handle", "source_post_text",
        "reply_count", "quote_count", "retweet_count", "like_count",
        "engagement_score", "planned_max_replies",
    ]
    coverage_df = selected_manifest_df[[c for c in coverage_cols if c in selected_manifest_df.columns]].copy()
    coverage_df = coverage_df.merge(actual_replies_df, on="source_post_id", how="left")
    coverage_df["n_replies_collected"] = pd.to_numeric(
        coverage_df["n_replies_collected"], errors="coerce"
    ).fillna(0).astype(int)
    coverage_df["n_reply_authors"] = pd.to_numeric(
        coverage_df["n_reply_authors"], errors="coerce"
    ).fillna(0).astype(int)

    stats_cols = ["source_post_id", "status", "status_code", "n_rows_kept", "seconds", "collection_batch_id"]
    stats_merge_df = reply_stats_df[[c for c in stats_cols if c in reply_stats_df.columns]].copy()
    coverage_df = coverage_df.merge(
        stats_merge_df,
        on=["source_post_id", "collection_batch_id"],
        how="left",
    )
    coverage_df["collection_state"] = "not_attempted"
    coverage_df.loc[coverage_df["status"].eq("ok"), "collection_state"] = "completed_ok"
    coverage_df.loc[coverage_df["status_code"].astype(str).eq("402"), "collection_state"] = "credits_depleted"
    coverage_df.loc[
        coverage_df["status"].notna()
        & ~coverage_df["status"].eq("ok")
        & ~coverage_df["status_code"].astype(str).eq("402"),
        "collection_state",
    ] = "other_error"
    coverage_df["reply_count"] = pd.to_numeric(coverage_df["reply_count"], errors="coerce").fillna(0)
    coverage_df["reply_collection_ratio"] = (
        coverage_df["n_replies_collected"]
        / coverage_df["reply_count"].replace(0, pd.NA)
    )
else:
    coverage_df = pd.DataFrame()

coverage_summary_df = (
    coverage_df.groupby("collection_state", dropna=False)
    .agg(
        source_posts=("source_post_id", "nunique"),
        reported_replies=("reply_count", "sum"),
        collected_replies=("n_replies_collected", "sum"),
    )
    .reset_index()
    if not coverage_df.empty else pd.DataFrame()
)

completed_coverage_df = coverage_df[coverage_df["collection_state"].eq("completed_ok")].copy()
problematic_collection_df = coverage_df[
    (coverage_df["reply_count"] >= 5)
    & (coverage_df["n_replies_collected"] <= 1)
].sort_values(["collection_state", "reply_count"], ascending=[True, False])

top_source_posts_df = coverage_df.sort_values(
    ["reply_count", "engagement_score"], ascending=False
).head(30)

eda.atomic_to_csv(coverage_df, REPORTS_DIR / "reply_collection_coverage.csv")
eda.atomic_to_csv(coverage_summary_df, REPORTS_DIR / "reply_collection_coverage_summary.csv")
eda.atomic_to_csv(problematic_collection_df, REPORTS_DIR / "problematic_reply_collection_posts.csv")
eda.atomic_to_csv(top_source_posts_df, REPORTS_DIR / "top_source_posts_by_reply_count.csv")

display(coverage_summary_df)
display(top_source_posts_df[[
    c for c in ["source_post_id", "event_id", "media_handle", "reply_count",
                "n_replies_collected", "reply_collection_ratio", "collection_state",
                "source_post_text"] if c in top_source_posts_df.columns
]].head(20))

if not completed_coverage_df.empty:
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(
        completed_coverage_df["reply_count"],
        completed_coverage_df["n_replies_collected"],
        alpha=0.55,
        color=COLORS["navy"],
        edgecolor="none",
    )
    maximum = max(
        float(completed_coverage_df["reply_count"].max()),
        float(completed_coverage_df["n_replies_collected"].max()),
        1,
    )
    ax.plot([0, maximum], [0, maximum], linestyle="--", color=COLORS["red"], linewidth=1)
    ax.set_xlabel("reply_count reportado por X")
    ax.set_ylabel("Replies recuperadas")
    ax.set_title("Cobertura efectiva por post madre completado")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "reply_collection_coverage.png", dpi=170)
    plt.close(fig)

## 7. Palabras y n-gramas frecuentes

In [ ]:
text_series = corpus_df["text_norm_no_accents"].fillna("")
top_words_df = eda.count_ngrams(text_series, n=1, top_n=TOP_N)
top_bigrams_df = eda.count_ngrams(text_series, n=2, top_n=TOP_N)
top_trigrams_df = eda.count_ngrams(text_series, n=3, top_n=TOP_N)

eda.atomic_to_csv(top_words_df, REPORTS_DIR / "top_words.csv")
eda.atomic_to_csv(top_bigrams_df, REPORTS_DIR / "top_bigrams.csv")
eda.atomic_to_csv(top_trigrams_df, REPORTS_DIR / "top_trigrams.csv")


def horizontal_top_plot(table, label_col, title, output_name, color):
    if table.empty:
        return
    part = table.head(25).sort_values("count")
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(part[label_col], part["count"], color=color)
    ax.set_title(title)
    ax.set_xlabel("Frecuencia")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / output_name, dpi=170)
    plt.close(fig)


horizontal_top_plot(top_words_df, "term", "Palabras más frecuentes", "top_words.png", COLORS["navy"])
horizontal_top_plot(top_bigrams_df, "2gram", "Bigramas más frecuentes", "top_bigrams.png", COLORS["teal"])
horizontal_top_plot(top_trigrams_df, "3gram", "Trigramas más frecuentes", "top_trigrams.png", COLORS["red"])

display(top_words_df.head(30))
display(top_bigrams_df.head(30))

## 8. Exploración preliminar con el lexicón

Las coincidencias del lexicón son indicadores de cobertura léxica, no clasificaciones de hostilidad ni discurso de odio. El lexicón actual es amplio y contiene numerosas entradas sin categoría; por eso se reportan por separado las coincidencias totales y las coincidencias con términos categorizados.

In [ ]:
corpus_eda_df, lexicon_metadata_df = eda.apply_lexicon_matches(
    corpus_df,
    lexicon_df,
    text_col="text_norm_no_accents",
)

lexicon_by_event_df = (
    corpus_eda_df.groupby(["event_id", "event_name"], dropna=False)
    .agg(
        n_rows=("tweet_id", "size"),
        any_lexicon_matches=("has_lexicon_match", "sum"),
        categorized_matches=("has_categorized_lexicon_match", "sum"),
        mean_hit_count=("lexicon_hit_count", "mean"),
    )
    .reset_index()
)
lexicon_by_event_df["any_match_pct"] = (
    lexicon_by_event_df["any_lexicon_matches"] / lexicon_by_event_df["n_rows"] * 100
).round(2)
lexicon_by_event_df["categorized_match_pct"] = (
    lexicon_by_event_df["categorized_matches"] / lexicon_by_event_df["n_rows"] * 100
).round(2)
lexicon_by_event_df["event_order"] = lexicon_by_event_df["event_id"].map(
    {event_id: index for index, event_id in enumerate(EVENT_ORDER)}
)
lexicon_by_event_df = lexicon_by_event_df.sort_values("event_order")

lexicon_by_media_df = (
    corpus_eda_df.groupby(["anchor_media_id", "anchor_media_handle"], dropna=False)
    .agg(
        n_rows=("tweet_id", "size"),
        any_lexicon_matches=("has_lexicon_match", "sum"),
        categorized_matches=("has_categorized_lexicon_match", "sum"),
    )
    .reset_index()
)
lexicon_by_media_df["any_match_pct"] = (
    lexicon_by_media_df["any_lexicon_matches"] / lexicon_by_media_df["n_rows"] * 100
).round(2)
lexicon_by_media_df["categorized_match_pct"] = (
    lexicon_by_media_df["categorized_matches"] / lexicon_by_media_df["n_rows"] * 100
).round(2)

term_counter = Counter()
for value in corpus_eda_df["lexicon_terms_found"].fillna(""):
    term_counter.update(term for term in str(value).split("|") if term)
top_lexicon_terms_df = pd.DataFrame(term_counter.most_common(200), columns=["term", "count"])

category_counter = Counter()
for value in corpus_eda_df["lexicon_categories_found"].fillna(""):
    category_counter.update(category for category in str(value).split("|") if category)
lexicon_category_counts_df = pd.DataFrame(category_counter.most_common(), columns=["category", "count"])

lexicon_summary_df = pd.DataFrame([
    {"metric": "lexicon_rows", "value": len(lexicon_df)},
    {"metric": "normalized_unique_terms_indexed", "value": len(lexicon_metadata_df)},
    {"metric": "corpus_rows", "value": len(corpus_eda_df)},
    {"metric": "rows_with_any_match", "value": int(corpus_eda_df["has_lexicon_match"].sum())},
    {"metric": "pct_with_any_match", "value": round(corpus_eda_df["has_lexicon_match"].mean() * 100, 2)},
    {"metric": "rows_with_categorized_match", "value": int(corpus_eda_df["has_categorized_lexicon_match"].sum())},
    {"metric": "pct_with_categorized_match", "value": round(corpus_eda_df["has_categorized_lexicon_match"].mean() * 100, 2)},
    {"metric": "methodological_status", "value": "exploratory_not_classification"},
])

eda.atomic_to_csv(lexicon_summary_df, REPORTS_DIR / "lexicon_summary.csv")
eda.atomic_to_csv(lexicon_by_event_df, REPORTS_DIR / "lexicon_by_event.csv")
eda.atomic_to_csv(lexicon_by_media_df, REPORTS_DIR / "lexicon_by_anchor_media.csv")
eda.atomic_to_csv(top_lexicon_terms_df, REPORTS_DIR / "top_lexicon_terms.csv")
eda.atomic_to_csv(lexicon_category_counts_df, REPORTS_DIR / "lexicon_category_counts.csv")
eda.atomic_to_csv(lexicon_metadata_df, REPORTS_DIR / "lexicon_index_metadata.csv")

display(lexicon_summary_df)
display(lexicon_by_event_df)

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(
    lexicon_by_event_df["event_name"],
    lexicon_by_event_df["categorized_match_pct"],
    color=COLORS["red"],
)
ax.set_title("Coincidencia con términos categorizados del lexicón")
ax.set_ylabel("Porcentaje de replies")
ax.tick_params(axis="x", rotation=35)
for label in ax.get_xticklabels():
    label.set_ha("right")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "categorized_lexicon_rate_by_event.png", dpi=170)
plt.close(fig)

## 9. Muestra estratificada para revisión manual

La plantilla utiliza tres etiquetas humanas complementarias:

- `hostility_relevance`: nivel `0`, `1`, `2` o `3`.
- `manual_hostility`: `0` no encontrada, `1` encontrada.
- `manual_hate_speech`: `0` no encontrado, `1` encontrado.
- `notes`: opcional.

La muestra existente nunca se sobrescribe. Si el archivo canónico ya existe, esta
celda escribe `manual_review_sample_candidate.csv`.


In [ ]:
corpus_eda_df["engagement_proxy"] = (
    corpus_eda_df["reply_count"]
    + corpus_eda_df["quote_count"]
    + corpus_eda_df["retweet_count"]
    + corpus_eda_df["like_count"]
)

sample_parts = []
for event_index, event_id in enumerate(EVENT_ORDER):
    part = corpus_eda_df[corpus_eda_df["event_id"].eq(event_id)].copy()
    if part.empty:
        continue
    target = min(MANUAL_SAMPLE_PER_EVENT, len(part))
    priority_n = min(target // 2, int(part["has_categorized_lexicon_match"].sum()))
    priority = part[part["has_categorized_lexicon_match"]].sort_values(
        ["categorized_lexicon_hit_count", "engagement_proxy"], ascending=False
    ).head(priority_n).copy()
    priority["sampling_reason"] = "lexicon_priority"
    chosen_indices = set(priority.index)

    engagement_n = min(target // 4, target - len(chosen_indices))
    engagement = part.loc[~part.index.isin(chosen_indices)].sort_values(
        "engagement_proxy", ascending=False
    ).head(engagement_n).copy()
    engagement["sampling_reason"] = "engagement_priority"
    chosen_indices.update(engagement.index)

    remaining = target - len(chosen_indices)
    pool = part.loc[~part.index.isin(chosen_indices)]
    random_part = pool.sample(
        n=min(remaining, len(pool)),
        random_state=RANDOM_STATE + event_index,
    ).copy() if remaining > 0 and not pool.empty else pool.head(0).copy()
    random_part["sampling_reason"] = "random_complement"

    selected = pd.concat([priority, engagement, random_part], ignore_index=False)
    sample_parts.append(selected)

manual_sample_df = (
    pd.concat(sample_parts, ignore_index=True, sort=False)
    .drop_duplicates(subset=["tweet_id"], keep="first")
    .reset_index(drop=True)
)
manual_sample_df["review_id"] = [f"formal_rev_{index + 1:04d}" for index in range(len(manual_sample_df))]
manual_sample_df["annotation_schema_version"] = "hatecr_4level_binary_v2"
manual_sample_df["hostility_relevance"] = pd.NA
manual_sample_df["manual_hostility"] = pd.NA
manual_sample_df["manual_hate_speech"] = pd.NA
manual_sample_df["notes"] = pd.NA

manual_columns = [
    "review_id", "annotation_schema_version", "sampling_reason", "tweet_id",
    "source_type", "event_id", "event_name", "anchor_media_id",
    "anchor_media_handle", "source_post_id", "created_at",
    "reply_author_id_hash", "text", "categorized_lexicon_hit_count",
    "lexicon_terms_found", "lexicon_categories_found",
    "hostility_relevance", "manual_hostility", "manual_hate_speech", "notes",
]
manual_sample_df = manual_sample_df[manual_columns]
manual_sample_path = REPORTS_DIR / "manual_review_sample.csv"

# Never overwrite the canonical file once it exists; it may be under active annotation.
if manual_sample_path.exists():
    manual_sample_path = REPORTS_DIR / "manual_review_sample_candidate.csv"
    print("[SAFE] La muestra canónica existe y no será modificada.")

eda.atomic_to_csv(manual_sample_df, manual_sample_path)
print("Muestra manual:", len(manual_sample_df), "filas")
print("Guardada en:", manual_sample_path)
display(manual_sample_df.groupby("event_id").size().reset_index(name="n_rows"))


## 10. Duplicados y valores faltantes

In [ ]:
duplicate_tweet_rows_df = corpus_eda_df[
    corpus_eda_df.duplicated("tweet_id", keep=False)
].copy()
duplicate_text_mask = corpus_eda_df.duplicated("text_norm_hash", keep=False)
duplicate_text_groups_df = (
    corpus_eda_df.loc[duplicate_text_mask]
    .groupby("text_norm_hash", dropna=False)
    .agg(
        n_rows=("tweet_id", "size"),
        n_unique_tweets=("tweet_id", "nunique"),
        n_unique_authors=("reply_author_id_hash", "nunique"),
        n_events=("event_id", "nunique"),
        n_anchor_media=("anchor_media_handle", "nunique"),
    )
    .reset_index()
    .sort_values(["n_rows", "n_unique_authors"], ascending=False)
)

key_columns = [
    "tweet_id", "text", "created_at", "reply_author_id_hash",
    "conversation_id", "source_type", "anchor_post_id", "anchor_media_handle",
    "event_id", "lang",
]
missing_values_df = pd.DataFrame([
    {
        "column": column,
        "missing_count": int(corpus_eda_df[column].isna().sum()) if column in corpus_eda_df.columns else len(corpus_eda_df),
        "missing_pct": round(
            (corpus_eda_df[column].isna().mean() * 100) if column in corpus_eda_df.columns else 100.0,
            2,
        ),
    }
    for column in key_columns
])

duplicate_summary_df = pd.DataFrame([
    {"metric": "duplicate_tweet_rows", "value": len(duplicate_tweet_rows_df)},
    {"metric": "duplicate_text_rows", "value": int(duplicate_text_mask.sum())},
    {"metric": "duplicate_text_groups", "value": len(duplicate_text_groups_df)},
])

eda.atomic_to_csv(duplicate_tweet_rows_df, REPORTS_DIR / "duplicates_by_tweet_id.csv")
eda.atomic_to_csv(duplicate_text_groups_df, REPORTS_DIR / "duplicates_by_text_norm_hash.csv")
eda.atomic_to_csv(duplicate_summary_df, REPORTS_DIR / "duplicates_summary.csv")
eda.atomic_to_csv(missing_values_df, REPORTS_DIR / "missing_values_diagnostics.csv")

display(duplicate_summary_df)
display(missing_values_df)

## 11. Exportación del corpus EDA enriquecido

In [ ]:
assert corpus_eda_df["tweet_id"].is_unique
assert corpus_eda_df["reply_author_id_hash"].str.fullmatch(r"[0-9a-f]{64}").all()
assert not {"author_id", "username", "screen_name"}.intersection(corpus_eda_df.columns)
assert corpus_eda_df["source_type"].eq("reply_to_media_post").all()
assert corpus_eda_df["anchor_media_handle"].notna().all()

eda.atomic_to_csv(corpus_eda_df, EDA_OUTPUT_PATH)
print("[OK] Corpus formal EDA:", EDA_OUTPUT_PATH)
print("Filas:", len(corpus_eda_df))

## 12. Conclusiones automáticas y preguntas de decisión

In [ ]:
top_event_row = by_event_df.sort_values("n_rows", ascending=False).iloc[0]
top_media_row = by_media_df.sort_values("n_rows", ascending=False).iloc[0]
coverage_state_counts = (
    coverage_df["collection_state"].value_counts().to_dict()
    if not coverage_df.empty else {}
)

auto_conclusions_df = pd.DataFrame([
    {"finding": "analysis_corpus_rows", "value": len(corpus_eda_df)},
    {"finding": "master_corpus_rows", "value": len(master_df)},
    {"finding": "training_dedup_rows", "value": len(training_df)},
    {"finding": "top_event", "value": top_event_row["event_id"]},
    {"finding": "top_event_rows", "value": int(top_event_row["n_rows"])},
    {"finding": "top_anchor_media", "value": top_media_row["anchor_media_handle"]},
    {"finding": "top_anchor_media_rows", "value": int(top_media_row["n_rows"])},
    {"finding": "lexicon_any_match_pct", "value": round(corpus_eda_df["has_lexicon_match"].mean() * 100, 2)},
    {"finding": "lexicon_categorized_match_pct", "value": round(corpus_eda_df["has_categorized_lexicon_match"].mean() * 100, 2)},
    {"finding": "selected_source_posts", "value": len(coverage_df)},
    {"finding": "source_posts_completed_ok", "value": int(coverage_state_counts.get("completed_ok", 0))},
    {"finding": "source_posts_credits_depleted", "value": int(coverage_state_counts.get("credits_depleted", 0))},
    {"finding": "source_posts_not_attempted", "value": int(coverage_state_counts.get("not_attempted", 0))},
    {"finding": "manual_review_sample_rows", "value": len(manual_sample_df)},
    {"finding": "methodological_warning", "value": "lexicon_matches_are_not_hate_speech_labels"},
])
eda.atomic_to_csv(auto_conclusions_df, REPORTS_DIR / "eda_auto_conclusions.csv")
display(auto_conclusions_df)

## 13. Advertencia metodológica

1. El corpus está anclado en posts de medios costarricenses y no representa toda la conversación política en X.
2. Solo contiene replies recuperadas hasta el agotamiento de créditos; quedan posts seleccionados sin consultar.
3. El volumen por medio depende de publicación, selección, engagement, disponibilidad y recuperación efectiva.
4. Una coincidencia de lexicón no equivale a hostilidad ni discurso de odio. El  elevado nivel de cobertura exige validación manual y revisión de términos sin categoría.
5. Las repeticiones textuales pueden indicar fórmulas comunes, coordinación o contenido automatizado; se preservan en el corpus descriptivo y se eliminan únicamente en la vista de entrenamiento.
6. Las comparaciones entre eventos deben considerar diferencias de ventana, número de posts madre y actividad pública.

### Preguntas para el siguiente paso
- ¿Qué eventos y medios concentran más replies analizables?
- ¿La cobertura incompleta cambia la distribución por evento o medio?
- ¿Qué categorías del lexicón requieren depuración antes de interpretar resultados?
- ¿La muestra manual de 180 textos es suficiente para una primera validación?